In [1]:
import os
import math
import random
from timeit import default_timer as timer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

from datasets import load_dataset, DatasetDict, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer

from tqdm.autonotebook import tqdm
import evaluate

/users/kent/student/bboggia/miniforge3/envs/pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("Using:", device)

Using: cuda


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [4]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision("high")

In [5]:
from concurrent.futures import ThreadPoolExecutor

def _load(args):
    return load_dataset(*args) if isinstance(args, tuple) else load_dataset(args)

with ThreadPoolExecutor(max_workers=5) as pool:
    ds1, ds2 = pool.map(_load, [
        "knkarthick/samsum",
        ("cnn_dailymail", "3.0.0")
    ])
    
def preprocess_dialogue(example):
    return {"text": example["dialogue"], "summary": example["summary"]}

def preprocess_cnn_dailymail(example):
    return {"text": example["article"], "summary": example["highlights"]}

NUM_PROC = 8

samsum = ds1.map(preprocess_dialogue, remove_columns=ds1["train"].column_names, num_proc=NUM_PROC)
cnn_dm = ds2.map(preprocess_cnn_dailymail, remove_columns=ds2["train"].column_names, num_proc=NUM_PROC)

dataset = DatasetDict({
    "train": concatenate_datasets([samsum["train"], cnn_dm["train"]]).shuffle(seed=42),
    "validation": concatenate_datasets([samsum["validation"], cnn_dm["validation"]]).shuffle(seed=42),
    "test": concatenate_datasets([samsum["test"], cnn_dm["test"]]).shuffle(seed=42),
})

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['summary', 'text'],
        num_rows: 301844
    })
    validation: Dataset({
        features: ['summary', 'text'],
        num_rows: 14186
    })
    test: Dataset({
        features: ['summary', 'text'],
        num_rows: 12309
    })
})
{'summary': "After speculation all weekend workers find out in ten minute meeting this morning .\nDavid Davis: 'They are an outstanding workforce and worth fighting for'\nAlan Johnson: 'It is terrible news delivered in a terrible way'\nMajority of jobs belonged to highly-skilled workers .", 'text': "After speculation all weekend workers find out in ten minute meeting this morning . David Davis: 'They are an outstanding workforce and worth fighting for' Alan Johnson: 'It is terrible news delivered in a terrible way' Majority of jobs belonged to highly-skilled workers . By . Simon Duke . Last updated at 4:49 PM on 27th September 2011 . Politicians and union bosses united today to condemn the treatm

In [6]:
def corpus_iterator(ds):
    splits = ["train", "validation", "test"]
    total_items = 2 * sum(len(ds[split]) for split in splits)

    with tqdm(total=total_items, desc="Tokenizer corpus", unit="text") as pbar:
        for split in splits:
            for ex in ds[split]:
                yield ex["text"]
                pbar.update(1)
                yield ex["summary"]
                pbar.update(1)

In [ ]:
vocab_size = 18000

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]

tokenizer = ByteLevelBPETokenizer()
corpus_size = 2 * sum(len(dataset[split]) for split in ["train", "validation", "test"])
# tokenizer.train_from_iterator(
#     corpus_iterator(dataset),
#     vocab_size=vocab_size,
#     min_frequency=2,
#     special_tokens=special_tokens,
#     length=corpus_size
# )

# # os.makedirs("tokenizer_class_ex", exist_ok=True)
# tokenizer.save_model("tokenizer_class_ex")
print("Tokenizer saved!")

Tokenizer corpus: 100%|██████████| 656678/656678 [03:57<00:00, 2761.25text/s]





Tokenizer saved!


In [8]:
tokenizer = ByteLevelBPETokenizer(
    "tokenizer_class_ex/vocab.json",
    "tokenizer_class_ex/merges.txt"
)

vocab_size = tokenizer.get_vocab_size()

pad_id = tokenizer.token_to_id("<pad>")
unk_id = tokenizer.token_to_id("<unk>")
bos_id = tokenizer.token_to_id("<bos>")
eos_id = tokenizer.token_to_id("<eos>")

print("Vocab size:", vocab_size)
print("Special IDs: ", {"pad": pad_id, "unk": unk_id, "bos": bos_id, "eos": eos_id})

Vocab size: 18000
Special IDs:  {'pad': 0, 'unk': 1, 'bos': 2, 'eos': 3}


In [9]:
max_src_len = 384
max_tgt_len = 160

d_model = 384
n_heads = 12
n_enc_layers = 6
n_dec_layers = 6
d_ff = 1024
dropout = 0.11

In [10]:
def encode_source(text: str):
    ids = tokenizer.encode(text).ids
    ids = ids[:max_src_len - 1]
    return ids + [eos_id]

In [11]:
def encode_target(text: str):
    ids = tokenizer.encode(text).ids
    ids = ids[:max_tgt_len - 2]
    return [bos_id] + ids + [eos_id]

In [12]:
def tokenize_ex(ex):
    return {
        "src_ids": encode_source(ex["text"]),
        "tgt_ids": encode_target(ex["summary"])
    }

tokenized = dataset.map(
    tokenize_ex,
    remove_columns=dataset["train"].column_names,
    num_proc=NUM_PROC
)

print(tokenized["train"][0].keys())
print(tokenized["train"][0]["src_ids"][:20])
print(tokenized["train"][0]["tgt_ids"][:20])

Map (num_proc=8): 100%|██████████| 12309/12309 [00:02<00:00, 4883.42 examples/s]

dict_keys(['src_ids', 'tgt_ids'])
[6018, 8486, 555, 2583, 2848, 1231, 497, 287, 2921, 3839, 2766, 499, 1918, 299, 1632, 5683, 29, 358, 1883, 407]
[2, 6018, 8486, 555, 2583, 2848, 1231, 497, 287, 2921, 3839, 2766, 499, 1918, 299, 202, 8518, 5683, 29, 358]


In [13]:
def mask_batch(batch):
    src_list = [torch.tensor(x["src_ids"], dtype=torch.long) for x in batch]
    tgt_list = [torch.tensor(x["tgt_ids"], dtype=torch.long) for x in batch]
    
    src = pad_sequence(src_list, batch_first=True, padding_value=pad_id)
    tgt = pad_sequence(tgt_list, batch_first=True, padding_value=pad_id)
    
    tgt_in = tgt[:, :-1] # [B, T-1]
    tgt_out = tgt[:, 1:]
    
    src_pad_mask = (src == pad_id)
    tgt_pad_mask = (tgt_in == pad_id)
    
    return src, tgt_in, tgt_out, src_pad_mask, tgt_pad_mask

In [14]:
train_batch_size = 128 # 64 for local running
eval_batch_size = 256 # 128 for local running

num_workers = 8 # local no more then 4 to be safe
pin_memory = (device.type == "cuda")

train_loader = DataLoader(
    tokenized["train"],
    batch_size = train_batch_size,
    shuffle = True,
    collate_fn= mask_batch,
    num_workers = num_workers,
    pin_memory = pin_memory,
    persistent_workers=(num_workers > 0),
    prefetch_factor = 4 if num_workers > 0 else None
)

val_loader = DataLoader(
    tokenized["validation"],
    batch_size = eval_batch_size,
    shuffle = False,
    collate_fn= mask_batch,
    num_workers = num_workers,
    pin_memory = pin_memory,
    persistent_workers=(num_workers > 0),
    prefetch_factor = 4 if num_workers > 0 else None
)

test_loader = DataLoader(
    tokenized["test"],
    batch_size = eval_batch_size,
    shuffle = False,
    collate_fn= mask_batch,
    num_workers = num_workers,
    pin_memory = pin_memory,
    persistent_workers=(num_workers > 0),
    prefetch_factor = 4 if num_workers > 0 else None
)

In [15]:
max_causal_len = max_tgt_len
full_causal_mask = torch.triu(
    torch.ones(max_causal_len, max_causal_len, dtype=torch.bool),
    diagonal=1
).to(device)

def get_causal_mask(T: int):
    if T <= full_causal_mask.size(0):
        return full_causal_mask[:T, :T] # [T, T]
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

In [16]:
class GEGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.proj = nn.Linear(d_model, 2 * d_ff)
        self.out = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        value, gate = self.proj(x).chunk(2, dim=-1)
        x = value * F.gelu(gate)
        x = self.dropout(x)
        return self.out(x)

In [17]:
class MHA(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            embed_dim = d_model,
            num_heads = n_heads,
            dropout = dropout,
            batch_first = True # [B, T, D]
        )
        
    def forward(self, q, k, v, key_padding_mask = None, attn_mask = None):
        out, _ = self.mha(
            q, k, v,
            key_padding_mask = key_padding_mask,
            attn_mask = attn_mask,
            need_weights=False
        )
        
        return out

In [18]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MHA(d_model, n_heads, dropout)
        self.drop1 = nn.Dropout(dropout)
        
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = GEGLU(d_model, d_ff, dropout)
        self.drop2 = nn.Dropout(dropout)
        
    def forward(self, x, src_pad_mask):
        h = self.ln1(x)
        x_attn = self.self_attn(h, h, h, key_padding_mask = src_pad_mask)
        x = x + self.drop1(x_attn)
        
        h = self.ln2(x)
        x_ff = self.ff(h)
        x = x + self.drop2(x_ff)
        
        return x

In [19]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MHA(d_model, n_heads, dropout)
        self.drop1 = nn.Dropout(dropout)
        
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MHA(d_model, n_heads, dropout)
        self.drop2 = nn.Dropout(dropout)
        
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = GEGLU(d_model, d_ff, dropout)
        self.drop3 = nn.Dropout(dropout)
        
    def forward(self, x, memory, tgt_pad_mask, src_pad_mask, causal_mask):
        h = self.ln1(x)
        x_attn = self.self_attn(
            h, h, h, 
            key_padding_mask = tgt_pad_mask,
            attn_mask = causal_mask
        )
        x = x + self.drop1(x_attn)
        
        h = self.ln2(x)
        x_cross_attn = self.cross_attn(
            h, memory, memory, 
            key_padding_mask = src_pad_mask,
        )
        x = x + self.drop2(x_cross_attn)
        
        h = self.ln3(x)
        x_ff = self.ff(h)
        x = x + self.drop3(x_ff)
        
        return x

In [20]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model = 256, n_heads = 8,
                 n_enc_layers = 4, n_dec_layers = 4,
                 d_ff = 768, dropout = 0.1,
                 max_src_len = 256, max_tgt_len = 96):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        
        self.src_pos_emb = nn.Embedding(max_src_len + 2, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len + 2, d_model)
        
        self.emb_drop = nn.Dropout(dropout)
        
        self.encoder = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)
        ])
        
        self.decoder = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)
        ])
        
        self.enc_ln = nn.LayerNorm(d_model)
        self.dec_ln = nn.LayerNorm(d_model)
        
        self.out_proj = nn.Linear(d_model, vocab_size, bias=False)
        
        self.out_proj.weight = self.tok_emb.weight
        
    def encode(self, src, src_pad_mask): 
        B, S = src.shape
        pos = torch.arange(S, device = src.device).unsqueeze(0).expand(B, S)
        
        x = self.tok_emb(src) * math.sqrt(self.d_model)
        x = x + self.src_pos_emb(pos)
        x = self.emb_drop(x)
        
        for layer in self.encoder:
            x = layer(x, src_pad_mask)
            
        return self.enc_ln(x)
    
    def decode(self, tgt_in, memory, tgt_pad_mask, src_pad_mask, causal_mask):
        B, T = tgt_in.shape
        pos = torch.arange(T, device=tgt_in.device).unsqueeze(0).expand(B, T)
        
        x = self.tok_emb(tgt_in) * math.sqrt(self.d_model)
        x = x + self.tgt_pos_emb(pos)
        x = self.emb_drop(x)
        
        for layer in self.decoder:
            x = layer(x, memory, tgt_pad_mask, src_pad_mask, causal_mask)
            
        x = self.dec_ln(x)
        
        return self.out_proj(x)
    
    def forward(self, src, tgt_in, src_pad_mask, tgt_pad_mask, causal_mask):
        memory = self.encode(src, src_pad_mask)
        logits = self.decode(tgt_in, memory, tgt_pad_mask, src_pad_mask, causal_mask)
        
        return logits

In [21]:
model = Seq2SeqTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_enc_layers=n_enc_layers,
    n_dec_layers=n_dec_layers,
    d_ff=d_ff,
    dropout=dropout,
    max_src_len=max_src_len,
    max_tgt_len=max_tgt_len,
).to(device)

print("Model parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")

Model parameters: 31.976448 M


In [22]:
def init_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight)
        
model.apply(init_weights)

loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1)

In [ ]:
adamw_kwargs = dict(lr=2e-4, betas=(0.9, 0.98), weight_decay=0.01)
try:
    if device.type == "cuda":
        adamw_kwargs["fused"] = True
    optimizer = torch.optim.AdamW(model.parameters(), **adamw_kwargs)
except TypeError:
    optimizer = torch.optim.AdamW(model.parameters(), **adamw_kwargs)

warmup_steps = 1000 # 4700 
def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    return max(0.01, 1.0 - (step - warmup_steps) / 50000) 

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [24]:
use_amp = (device.type == "cuda")
amp_dtype = torch.bfloat16  
max_grad_norm = 1.0

In [25]:
def train_one_epoch(model, loader, scheduler=None):
    model.train()

    total_loss = 0.0
    total_tokens = 0  

    for src, tgt_in, tgt_out, src_pad_mask, tgt_pad_mask in tqdm(loader, leave=False):
        src = src.to(device, non_blocking=True)
        tgt_in = tgt_in.to(device, non_blocking=True)
        tgt_out = tgt_out.to(device, non_blocking=True)
        src_pad_mask = src_pad_mask.to(device, non_blocking=True)
        tgt_pad_mask = tgt_pad_mask.to(device, non_blocking=True)

        T = tgt_in.size(1)
        causal_mask = get_causal_mask(T)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask, causal_mask)
            loss = loss_fn(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        n_tokens = (tgt_out != pad_id).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

    return total_loss / max(total_tokens, 1)

In [26]:
@torch.inference_mode()
def eval_loss(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for src, tgt_in, tgt_out, src_pad_mask, tgt_pad_mask in loader:
        src = src.to(device, non_blocking=True)
        tgt_in = tgt_in.to(device, non_blocking=True)
        tgt_out = tgt_out.to(device, non_blocking=True)
        src_pad_mask = src_pad_mask.to(device, non_blocking=True)
        tgt_pad_mask = tgt_pad_mask.to(device, non_blocking=True)

        T = tgt_in.size(1)
        causal_mask = get_causal_mask(T)

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(src, tgt_in, src_pad_mask, tgt_pad_mask, causal_mask)
            loss = loss_fn(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))

        n_tokens = (tgt_out != pad_id).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

    return total_loss / max(total_tokens, 1)

In [27]:
@torch.inference_mode()
def greedy_generate(model, src, src_pad_mask, max_new_tokens=90, min_new_tokens=1):
    model.eval()
    src = src.to(device)
    src_pad_mask = src_pad_mask.to(device)

    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        memory = model.encode(src, src_pad_mask)

    B = src.size(0)
    ys = torch.full((B, 1), bos_id, dtype=torch.long, device=device)

    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for step in range(max_new_tokens):
        T = ys.size(1)
        causal_mask = get_causal_mask(T)

        tgt_pad_mask = (ys == pad_id)

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model.decode(ys, memory, tgt_pad_mask, src_pad_mask, causal_mask)

        next_token = logits[:, -1, :].argmax(dim=-1)

        if step + 1 < min_new_tokens:
            next_token = torch.where(next_token == eos_id, torch.full_like(next_token, unk_id), next_token)

        ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)

        finished |= (next_token == eos_id)
        if finished.all():
            break

    return ys

In [28]:
def decode_ids(ids):
    return tokenizer.decode(ids, skip_special_tokens=True)

def batch_decode(batch_ids):
    return [decode_ids(x.tolist()) for x in batch_ids]

In [29]:
rouge = evaluate.load("rouge")

@torch.inference_mode()
def eval_rouge(model, loader, max_batches=20, gen_max_new_tokens=max_tgt_len - 1, gen_min_new_tokens=8):
    model.eval()
    preds, refs = [], []
    pred_lens = []

    for b_idx, (src, _, tgt_out, src_pad_mask, _) in enumerate(loader):
        if b_idx >= max_batches:
            break

        gen_ids = greedy_generate(
            model,
            src,
            src_pad_mask,
            max_new_tokens=gen_max_new_tokens,
            min_new_tokens=gen_min_new_tokens,
        )
        pred_texts = batch_decode(gen_ids)
        ref_texts = batch_decode(tgt_out)

        pred_texts = [t.strip() for t in pred_texts]
        ref_texts = [t.strip() for t in ref_texts]

        preds.extend(pred_texts)
        refs.extend(ref_texts)
        pred_lens.extend([len(t.split()) for t in pred_texts])

    scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    scores["avg_pred_words"] = float(sum(pred_lens) / max(len(pred_lens), 1))
    scores["empty_pred_rate"] = float(sum(1 for t in preds if len(t) == 0) / max(len(preds), 1))

    return scores

In [30]:
import os

epochs = 20
start_epoch = 1
best_val = float("inf")

ckpt_best = "custom_summarizer_best.pt"
ckpt_latest = "custom_summarizer_latest.pt"

load_path = ckpt_latest if os.path.exists(ckpt_latest) else (ckpt_best if os.path.exists(ckpt_best) else None)

if load_path:
    print(f"Loading existing checkpoint from {load_path}...")
    ckpt = torch.load(load_path, map_location=device)
    
    state_dict = {k.removeprefix("_orig_mod."): v for k, v in ckpt["model_state"].items()}
    model.load_state_dict(state_dict)
    
    if "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    if "epoch" in ckpt:
        start_epoch = ckpt["epoch"] + 1
    if "best_val" in ckpt:
        best_val = ckpt["best_val"]
        
    print(f"Resuming from epoch {start_epoch} with best_val {best_val:.4f}")

start = timer()

for epoch in range(start_epoch, epochs + 1):
    train_loss = train_one_epoch(model, train_loader, scheduler=scheduler)
    val_loss = eval_loss(model, val_loader)

    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

    if epoch % 2 == 0:
        scores = eval_rouge(model, val_loader, max_batches=10, gen_max_new_tokens=max_tgt_len - 1)
        print("ROUGE subset:", {k: round(float(v), 4) for k, v in scores.items()})

    save_state = {
        "epoch": epoch,
        "best_val": best_val,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "config": {
            "vocab_size": vocab_size,
            "d_model": d_model,
            "n_heads": n_heads,
            "n_enc_layers": n_enc_layers,
            "n_dec_layers": n_dec_layers,
            "d_ff": d_ff,
            "dropout": dropout,
            "max_src_len": max_src_len,
            "max_tgt_len": max_tgt_len,
        },
        "special_ids": {"pad_id": pad_id, "bos_id": bos_id, "eos_id": eos_id, "unk_id": unk_id},
    }

    torch.save(save_state, ckpt_latest)

    if val_loss < best_val:
        best_val = val_loss
        save_state["best_val"] = best_val
        torch.save(save_state, ckpt_best)
        print(f"  Saved best checkpoint (val loss {best_val:.4f})")

end = timer()
if device.type == "cuda":
    torch.cuda.synchronize()
print(f"Training time: {end - start:.1f}s on {device}")

Epoch 01 | train loss 7.3025 | val loss 6.1060
  Saved best checkpoint (val loss 6.1060)


Epoch 02 | train loss 5.7605 | val loss 5.2390
ROUGE subset: {'rouge1': 0.0865, 'rouge2': 0.0117, 'rougeL': 0.0696, 'rougeLsum': 0.0822, 'avg_pred_words': 98.4727, 'empty_pred_rate': 0.0}
  Saved best checkpoint (val loss 5.2390)


Epoch 03 | train loss 5.0436 | val loss 4.6563
  Saved best checkpoint (val loss 4.6563)


Epoch 04 | train loss 4.6632 | val loss 4.4179
ROUGE subset: {'rouge1': 0.1418, 'rouge2': 0.0295, 'rougeL': 0.1005, 'rougeLsum': 0.1323, 'avg_pred_words': 106.9367, 'empty_pred_rate': 0.0}
  Saved best checkpoint (val loss 4.4179)


Epoch 05 | train loss 4.4603 | val loss 4.2642
  Saved best checkpoint (val loss 4.2642)


...

Epoch 10 | train loss 3.9828 | val loss 3.8882
ROUGE subset: {'rouge1': 0.1741, 'rouge2': 0.0471, 'rougeL': 0.1147, 'rougeLsum': 0.1602, 'avg_pred_words': 104.8152, 'empty_pred_rate': 0.0}
  Saved best checkpoint (val loss 3.8882)


Epoch 11 | train loss 3.9299 | val loss 3.8522
  Saved best checkpoint (val loss 3.8522)


Epoch 12 | train loss 3.8839 | val loss 3.8186
ROUGE subset: {'rouge1': 0.1801, 'rouge2': 0.0505, 'rougeL': 0.1182, 'rougeLsum': 0.1646, 'avg_pred_words': 105.5918, 'empty_pred_rate': 0.0}
  Saved best checkpoint (val loss 3.8186)


Epoch 13 | train loss 3.8441 | val loss 3.7917
  Saved best checkpoint (val loss 3.7917)


KeyboardInterrupt: 

In [31]:
ckpt = torch.load("custom_summarizer_best.pt", map_location=device)
state_dict = {k.removeprefix("_orig_mod."): v for k, v in ckpt["model_state"].items()}
model.load_state_dict(state_dict)
model = torch.compile(model)
model.eval()

OptimizedModule(
  (_orig_mod): Seq2SeqTransformer(
    (tok_emb): Embedding(18000, 384)
    (src_pos_emb): Embedding(386, 384)
    (tgt_pos_emb): Embedding(162, 384)
    (emb_drop): Dropout(p=0.11, inplace=False)
    (encoder): ModuleList(
      (0-5): 6 x EncoderLayer(
        (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (self_attn): MHA(
          (mha): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
          )
        )
        (drop1): Dropout(p=0.11, inplace=False)
        (ln2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (ff): GEGLU(
          (proj): Linear(in_features=384, out_features=2048, bias=True)
          (out): Linear(in_features=1024, out_features=384, bias=True)
          (dropout): Dropout(p=0.11, inplace=False)
        )
        (drop2): Dropout(p=0.11, inplace=False)
      )
    )
    (decoder): ModuleList(
      (0-5): 6 x DecoderLayer(
        (

In [32]:
test_loss = eval_loss(model, test_loader)
print("Test loss:", test_loss)

Test loss: 3.8088094607435234


In [33]:
ex = dataset["test"][0]
print("INPUT TEXT:\n", ex["text"][:1200])
print("\nREFERENCE SUMMARY:\n", ex["summary"] )

src_ids = torch.tensor([encode_source(ex["text"])], dtype=torch.long)
src_pad_mask = (src_ids == pad_id)

gen_ids = greedy_generate(model, src_ids, src_pad_mask, max_new_tokens=max_tgt_len - 1)
print("\nMODEL SUMMARY:\n", decode_ids(gen_ids[0].tolist()))

INPUT TEXT:
 Real Sociedad remain hopeful over a deal for Burnley striker Danny Ings. Liverpool, Manchester City and Manchester United have all shown interest in the England Under 21 international but have yet to sign him up as they consider other options also. Ings has interest from Borussia Monchengladbach among others but Sociedad have remained constant from the start. Real Sociedad are still hopeful of singing Burnley striker Danny Ings when his contract expires in the summer . Michel Vorm saves from Ings as the Burnley forward passes up a great chance to give his side the lead against Tottenham on Sunday . England Under 21 international Ings chats with Harry Kane after the final whistle at Turf Moor . The 23-year-old knows he can play regularly there, will be watched with interest by England coach Roy Hodgson and will get the money he wants. Sociedad manager David Moyes has taken the club into the top half of La Liga and intends to build on that in the summer and Ings could prove 

In [34]:
test_indices = random.sample(range(len(dataset["test"])), 5)

for i, idx in enumerate(test_indices, 1):
    ex = dataset["test"][idx]
    print(f"{'='*90}")
    print(f"EXAMPLE {i} (index {idx})")
    print(f"{'='*90}")
    print(f"INPUT TEXT (first 400 chars):\n{ex['text'][:500]}...\n")
    print(f"REFERENCE SUMMARY:\n{ex['summary']}\n")

    src_ids = torch.tensor([encode_source(ex["text"])], dtype=torch.long)
    src_pad_mask = (src_ids == pad_id)

    gen_ids = greedy_generate(model, src_ids, src_pad_mask, max_new_tokens=max_tgt_len - 1)
    print(f"MODEL SUMMARY:\n{decode_ids(gen_ids[0].tolist())}\n")

EXAMPLE 1 (index 10476)
INPUT TEXT (first 400 chars):
A nine-year-old girl was attacked by a lion that was being walked like a dog by circus entertainers who had brought their show to a quiet Siberian village. The girl is in hospital after she was savaged by the animal in Namtsy, in the Sakha Republic in Siberia - one of the coldest regions in the world. Photographs show the lion being walked around by a worker for a circus that had been visiting the village. A lion that appeared to be being walked like a dog around the Siberian village of Namtsy a...

REFERENCE SUMMARY:
A circus worker was seen walking a lion around a village like a dog .
The animal was seen being pulled by a makeshift leash in Namtsy, Siberia .
It savaged a nine-year-old girl in the village, on her way home from lessons .
Girl is in hospital and the extent of her injuries from the attack not known .

MODEL SUMMARY:
<bos>The girl was attacked by a circus dog in the Siberian village of Namytsha .
The animal was taken t

In [35]:
custom_texts = [
    "Amanda: Hey, are you coming to the party tonight?\nBen: I don't think so, I have a deadline tomorrow.\nAmanda: That's too bad! Everyone will miss you.\nBen: I know, but work comes first. Maybe next time?\nAmanda: Sure, good luck with your project!",

    "Scientists at MIT have developed a new algorithm that can predict protein structures with 95% accuracy. "
    "The breakthrough, published in Nature, could accelerate drug discovery by reducing the time needed to "
    "understand protein folding from months to hours. Lead researcher Dr. Sarah Chen said the method combines "
    "deep learning with physics-based simulations. Pharmaceutical companies have already expressed interest "
    "in licensing the technology.",

    "The city council voted 7-2 on Tuesday to approve a $50 million budget for infrastructure repairs, "
    "including road resurfacing, bridge maintenance, and water pipe replacements. Councilwoman Maria Lopez "
    "argued the investment was overdue, citing a 2023 report that rated 40% of city roads as 'poor condition.' "
    "Opponents said the spending would require a property tax increase of approximately 3%.",
]

for i, text in enumerate(custom_texts, 1):
    print(f"{'='*90}")
    print(f"CUSTOM EXAMPLE {i}")
    print(f"{'='*90}")
    print(f"INPUT:\n{text}\n")

    src_ids = torch.tensor([encode_source(text)], dtype=torch.long)
    src_pad_mask = (src_ids == pad_id)

    gen_ids = greedy_generate(model, src_ids, src_pad_mask, max_new_tokens=max_tgt_len - 1)
    print(f"MODEL SUMMARY:\n{decode_ids(gen_ids[0].tolist())}\n")

CUSTOM EXAMPLE 1
INPUT:
Amanda: Hey, are you coming to the party tonight?
Ben: I don't think so, I have a deadline tomorrow.
Amanda: That's too bad! Everyone will miss you.
Ben: I know, but work comes first. Maybe next time?
Amanda: Sure, good luck with your project!

MODEL SUMMARY:
<bos>Amanda is coming to the party tonight. Ben will come to the party.<eos>

CUSTOM EXAMPLE 2
INPUT:
Scientists at MIT have developed a new algorithm that can predict protein structures with 95% accuracy. The breakthrough, published in Nature, could accelerate drug discovery by reducing the time needed to understand protein folding from months to hours. Lead researcher Dr. Sarah Chen said the method combines deep learning with physics-based simulations. Pharmaceutical companies have already expressed interest in licensing the technology.

MODEL SUMMARY:
<bos>Dr Sarah Chenad says the method combines deep learning physics with physics .
The method combines deep learning physics with deep learning .
The techn